# House Price Prediction – End-to-End Implementation

This notebook demonstrates the end-to-end Machine Learning and MLOps workflow for predicting house prices using the Ames Housing dataset.

## 1. Load and Inspect Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load raw dataset
train_df = pd.read_csv('../data/raw/train.csv')
test_df = pd.read_csv('../data/raw/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
train_df.head()

In [ ]:
# Target Distribution Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train_df['SalePrice'], kde=True, ax=axes[0], color='royalblue')
axes[0].set_title('Original SalePrice (Right-Skewed)')

sns.histplot(np.log1p(train_df['SalePrice']), kde=True, ax=axes[1], color='forestgreen')
axes[1].set_title('Log-Transformed SalePrice (Normal)')
plt.tight_layout()
plt.show()

## 2. Preprocessing & Outlier Handling

In [ ]:
# Outlier Removal (GrLivArea > 4000 & SalePrice < 300000)
outliers = (train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)
print(f"Removing {outliers.sum()} outliers...")
train_df = train_df[~outliers].copy()
print(f"Cleaned Train shape: {train_df.shape}")

## 3. Train-Test Split, Feature Engineering & Artifact Export

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
import joblib
import os

from src.preprocess import engineer_features

y = np.log1p(train_df['SalePrice'])
X = train_df.drop(['Id', 'SalePrice'], axis=1)
X_eng = engineer_features(X)

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X_eng, y, test_size=0.2, random_state=42)

cat_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = X_train.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()

scaler = RobustScaler()
x_train_scaled = scaler.fit_transform(X_train[num_cols])
x_test_scaled = scaler.transform(X_test[num_cols])

ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
x_train_encoded = ohe.fit_transform(X_train[cat_cols])
x_test_encoded = ohe.transform(X_test[cat_cols])

X_train_final = np.hstack((x_train_scaled, x_train_encoded))
X_test_final = np.hstack((x_test_scaled, x_test_encoded))

print(f"X_train shape: {X_train_final.shape}")
print(f"X_test shape: {X_test_final.shape}")

## 4. Baseline Model Development & Evaluation

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

models = {
    'Ridge Baseline': Ridge(alpha=10.0, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.7, random_state=42)
}

results = []
for name, m in models.items():
    m.fit(X_train_final, y_train)
    preds = m.predict(X_test_final)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    preds_dollars = np.expm1(preds)
    y_dollars = np.expm1(y_test)
    rmse_dollars = np.sqrt(mean_squared_error(y_dollars, preds_dollars))
    
    results.append({
        'Model': name,
        'Log RMSE': rmse,
        'Log MAE': mae,
        'R2 Score': r2,
        'Dollar RMSE ($)': rmse_dollars
    })

results_df = pd.DataFrame(results).sort_values('Log RMSE')
display(results_df)

## 5. Model Evaluation & Diagnostic Visualizations

In [ ]:
best_model = models['XGBoost']
best_preds_log = best_model.predict(X_test_final)
best_preds_dollars = np.expm1(best_preds_log)
y_test_dollars = np.expm1(y_test)

# Actual vs Predicted Plot
plt.figure(figsize=(7, 6))
plt.scatter(y_test_dollars, best_preds_dollars, alpha=0.5, color='teal', edgecolors='k')
min_v, max_v = min(y_test_dollars.min(), best_preds_dollars.min()), max(y_test_dollars.max(), best_preds_dollars.max())
plt.plot([min_v, max_v], [min_v, max_v], 'r--', lw=2)
plt.title('Actual vs Predicted SalePrice ($) - XGBoost')
plt.xlabel('Actual SalePrice ($)')
plt.ylabel('Predicted SalePrice ($)')
plt.show()

In [ ]:
# Residuals Distribution
residuals = y_test - best_preds_log
plt.figure(figsize=(7, 5))
sns.histplot(residuals, kde=True, color='crimson')
plt.axvline(0, color='black', linestyle='--')
plt.title('Residuals Distribution (Log Scale)')
plt.xlabel('Residual Error')
plt.show()

## 6. Kaggle Submission Generation

In [ ]:
if os.path.exists('../data/processed/X_submission_final.npy') and os.path.exists('../data/processed/test_ids.npy'):
    X_sub = np.load('../data/processed/X_submission_final.npy')
    test_ids = np.load('../data/processed/test_ids.npy')
    sub_preds = np.expm1(best_model.predict(X_sub))
    
    submission = pd.DataFrame({'Id': test_ids, 'SalePrice': sub_preds})
    submission.to_csv('../outputs/submission.csv', index=False)
    print("Generated submission.csv:")
    display(submission.head())